# TradeSense · Component 03 · notebook 46: engine export (CPU, no training, no TEST)

**What.** Packs everything the live engine needs into one clean folder, `models_b2_v1/engine_export_v1/`, which you then copy into the repo (`component_03/backend/`):
- `models/` frozen weights per symbol (AE-ctx 3 seeds, XGBoost 3 boosters, Transformer 3 nets), N0 statistics, AE error scale. No training folders, no pickles.
- `config/` fusion v0 (stacker numbers as JSON, DEV genuine statistics), thresholds q (Alert 2/day, Watch-4, Watch-8), declared files of every protocol step, engine defaults.
- `code/` the frozen inference library (`c03_infer.py`, `b2_lib.py`) and the export loader (`c03_export.py`).
- `replay/` 10 days of 1-minute Layer 1 features (2024-08-01..08-10, includes the 2024-08-05 crash) with the expected scores, used as golden vectors and as the history-replay demo.
- `manifest.json` with sha256 of every file and the protocol / TEST hashes.

**Checks (all must pass).** The bundle loaded from the export folder reproduces the bundle loaded from the training folders bar for bar; the JSON logistic stacker equals the joblib stacker; the library is the same file that opened TEST (sha256 equals the one recorded in `test_b_results_v1.json`); no TEST key is read here.

In [ ]:
import os, sys, json, time, hashlib, warnings, datetime, glob, shutil, subprocess
from pathlib import Path
import numpy as np, pandas as pd
warnings.filterwarnings('ignore')
try:
    from google.colab import drive
    drive.mount('/content/drive')
except ImportError:
    print('Not in Colab -> using TRADESENSE_BASE.')
try:
    import xgboost as xgb
except ImportError:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'xgboost']); import xgboost as xgb
import torch, joblib
BASE = Path(os.environ.get('TRADESENSE_BASE', '/content/drive/MyDrive/research datasets'))
L3 = BASE/'layer3_v3'; OUT = BASE/'models_b2_v1'; FUS = OUT/'fusion'; EXP = OUT/'engine_export_v1'
SYMBOLS = ['BTCUSDT', 'ETHUSDT']; MOCK = os.environ.get('TS_MOCK') == '1'
LINES = []
def say(s=''): print(s); LINES.append(str(s))
def sha_file(p, n=16): return hashlib.sha256(Path(p).read_bytes()).hexdigest()[:n]
hf = L3/'protocol_bar_models_v2.sha256'; pf = L3/'protocol_bar_models_v2.md'; ph = hashlib.sha256(pf.read_text().encode()).hexdigest(); assert ph == hf.read_text().split()[0], 'frozen protocol B2 was changed'
for f in ['test_opened_v1.json', 'test_b_results_v1.json', 'declared_final_set_v1.json', 'declared_ae_v2.json', 'declared_fusion_v0.json', 'declared_watch_v1.json', 'c03_infer.py', 'b2_lib.py']:
    assert (OUT/f).exists(), f'{f} missing: run notebook 32 to the end first (TEST must be opened before the engine is packed)'
RES = json.loads((OUT/'test_b_results_v1.json').read_text()); assert sha_file(OUT/'c03_infer.py') == RES['library_sha16'], 'c03_infer.py differs from the version that opened TEST'
EXP.mkdir(exist_ok=True)
for sub in ['models', 'config', 'code', 'replay', 'docs']: (EXP/sub).mkdir(exist_ok=True)
sys.path.insert(0, str(OUT)); import importlib; import b2_lib as B; import c03_infer as c
say('protocol B2 intact, sha256 ' + ph[:16] + '; library sha256 ' + sha_file(OUT/'c03_infer.py') + ' (equals the TEST-opening library); TEST opened ' + RES['opened_utc'] + ('  [MOCK RUN]' if MOCK else ''))

Mounted at /content/drive
protocol B2 intact, sha256 443f99d5faf5929f; library sha256 3b5d9df23521adf9 (equals the TEST-opening library); TEST opened 2026-10-15T10:27:18Z


## 1 · Export weights, config, code

In [ ]:
open('c03_export.py', 'w').write(r'''"""TradeSense C03 - export loader: builds the frozen bar models from the engine export folder (no training directories, no pickles)."""
import json, numpy as np, torch
from pathlib import Path
import xgboost as xgb
import b2_lib as B
import c03_infer as c


class JsonLR:
    """The fusion v0 logistic stacker from its declared numbers: sigmoid input = ((x - mean) / scale) . coef + intercept."""
    def __init__(self, d): self.mean, self.scale, self.coef, self.intercept = (np.asarray(d['scaler_mean']), np.asarray(d['scaler_scale']), np.asarray(d['coef']), float(d['intercept']))
    def decision_function(self, X): return ((np.asarray(X, 'float64') - self.mean) / self.scale) @ self.coef + self.intercept
    def contributions(self, X): return (np.asarray(X, 'float64') - self.mean[None, :]) * (self.coef / self.scale)[None, :]


def load_bundle_from_export(root, sym, device='cpu'):
    root = Path(root); man = json.loads((root / 'manifest.json').read_text()); m = man['models'][sym]
    n0 = {k: np.asarray(v) for k, v in json.loads((root / m['n0']).read_text()).items()}; scale = np.asarray(json.loads((root / m['ae_scale']).read_text()))
    ae = []
    for p in m['ae']:
        n = B.AECtx(); n.load_state_dict(torch.load(root / p, map_location='cpu')); n.eval(); ae.append(n)
    xb = []
    for p, it in zip(m['xgb'], m['xgb_best_iteration']):
        b_ = xgb.Booster(); b_.load_model(str(root / p)); xb.append((b_, it))
    W, d, L = m['transformer_cfg']; nets = []
    for p in m['transformer']:
        n = c.TNet(W, d, L); n.load_state_dict(torch.load(root / p, map_location='cpu')); n.eval(); nets.append(n.to(device))
    return c.Bundle(sym, n0, man['modes'], ae, scale, xb, nets, W, device=device)


def load_fusion(root):
    root = Path(root); cfg = json.loads((root / 'config' / 'fusion.json').read_text())
    return JsonLR(cfg['logistic']), {s: {k: tuple(v) for k, v in d.items()} for s, d in cfg['gen_stats'].items()}, cfg
''')

sys.path.insert(0, '.'); import c03_export as X; importlib.reload(X)
DF = json.load(open(OUT/'declared_final_set_v1.json')); AE2 = json.load(open(OUT/'declared_ae_v2.json')); FUSD = json.load(open(OUT/'declared_fusion_v0.json')); WATCH = json.load(open(OUT/'declared_watch_v1.json'))
MODELS = c.MODEL_NAMES; MODES = {m: DF['models'][m]['normalisation'] for m in MODELS}; DEV_LO, DEV_HI = 90720, 129600; L1V = B.valid_mask(129600, DEV_LO, DEV_HI)
def find1(pat):
    c_ = [x for x in glob.glob(str(BASE/'**'/pat), recursive=True) if 'backup' not in x.lower()]; assert c_, pat; return sorted(c_)[0]
N0 = {}; AEC = {}; BUN0 = {}; GST = {}
for s in SYMBOLS:
    a = pd.read_parquet(find1(f'{s.lower()}_l1_feature_matrix_90d.parquet')); a = a.set_index('datetime') if 'datetime' in a.columns else a
    N0[s] = B.fit_n0(a[B.FEATS].reset_index(drop=True).iloc[15:90720]); AEC[s] = np.load(OUT/'scores'/f'ae_ctx_{s}_N1.npz')
    BUN0[s] = c.load_bundle_from_training(OUT, s, DF, AEC[s]['scale'], N0[s], 'cpu')
    zs = {}
    for m, fn in {MODELS[0]: 'ae_ctx_agg', MODELS[1]: 'xgb_v2', MODELS[2]: 'transformer_v2'}.items():
        v = np.load(OUT/'scores'/f'{fn}_{s}_{MODES[m]}.npz')['GEN_L1']; zs[m] = c.nanw(v[:, 0] if v.ndim == 2 else v)
    GST[s] = c.gen_stats(zs, L1V)
cx = DF['models'][MODELS[1]]; fs_, scope_, dtxt_ = cx['configuration'].split('|'); depth_ = int(dtxt_[1:]); ct = DF['models'][MODELS[2]]; cfgT = ct['configuration']; Wt, dt, Lt = [int(x[1:]) for x in cfgT.split('_')]
MAN = {'models': {}, 'modes': MODES}
for s in SYMBOLS:
    d = EXP/'models'/s; d.mkdir(exist_ok=True); b = BUN0[s]; ent = {'ae': [], 'xgb': [], 'xgb_best_iteration': [], 'transformer': []}
    for sd, n in zip(c.SEEDS, b.ae_nets): torch.save(n.state_dict(), d/f'ae_s{sd}.pt'); ent['ae'].append(f'models/{s}/ae_s{sd}.pt')
    for sd, (bo, it) in zip(cx['seeds'], b.xgb): bo.save_model(str(d/f'xgb_s{sd}.json')); ent['xgb'].append(f'models/{s}/xgb_s{sd}.json'); ent['xgb_best_iteration'].append(int(it))
    for sd, n in zip(ct['seeds'], b.tr_nets): torch.save(n.state_dict(), d/f'transformer_s{sd}.pt'); ent['transformer'].append(f'models/{s}/transformer_s{sd}.pt')
    (d/'n0.json').write_text(json.dumps({k: np.asarray(v).tolist() for k, v in N0[s].items()})); (d/'ae_scale.json').write_text(json.dumps(np.asarray(AEC[s]['scale']).tolist()))
    ent.update({'n0': f'models/{s}/n0.json', 'ae_scale': f'models/{s}/ae_scale.json', 'transformer_cfg': [Wt, dt, Lt], 'xgb_config': cx['configuration']}); MAN['models'][s] = ent
cfgd = EXP/'config'
(cfgd/'fusion.json').write_text(json.dumps({'logistic': FUSD['logistic'], 'inputs': FUSD['inputs'], 'q_T1_by_budget': FUSD['q_T1_by_budget'], 'gen_stats': {s: {m: list(v) for m, v in GST[s].items()} for s in SYMBOLS}, 'scaling': FUSD['scaling']}, indent=1))
assert FUSD['stacker']['kind'] == 'LR', 'the engine exports the logistic stacker only'
for f in ['declared_final_set_v1.json', 'declared_ae_v2.json', 'declared_fusion_v0.json', 'declared_watch_v1.json', 'declared_multihorizon_v2.json', 'declared_uncertainty_v1.json', 'declared_explain_v1.json', 'test_b_results_v1.json']:
    if (OUT/f).exists(): shutil.copy(OUT/f, cfgd/f)
ENGCFG = {'symbols': SYMBOLS, 'interval': '1m', 'warmup_bars': 10080, 'min_history_bars': 1440, 'state_rule': 'T1 on the live fused score (rolling 7-day own-score quantile)', 'alert_q': WATCH['alert']['q'], 'watch_q': WATCH['watch']['q'], 'watch_default_budget': '8', 'alert_budget_per_day': 2,
          'text': 'WATCH is amber and never an alert. Volume-burst-like behaviour is shown as WATCH with a caveat (ambiguous at 1-minute level). unusualness and integrity_score = 100 - unusualness are defined by declared_multihorizon_v2.json (percentile versus genuine DEV).', 'tgnn': None}
(cfgd/'engine_config.json').write_text(json.dumps(ENGCFG, indent=1))
for f in ['c03_infer.py', 'b2_lib.py']: shutil.copy(OUT/f, EXP/'code'/f)
Path(EXP/'code'/'c03_export.py').write_text(Path('c03_export.py').read_text())
if (OUT/'protocol_test_b_v1.md').exists(): pass
say('exported weights, N0 statistics, AE scales, fusion JSON, declared files and library to ' + str(EXP))

exported weights, N0 statistics, AE scales, fusion JSON, declared files and library to /content/drive/MyDrive/research datasets/models_b2_v1/engine_export_v1


## 2 · Verify the export

In [ ]:
MAN_TMP = EXP/'manifest.json'; MAN_TMP.write_text(json.dumps(MAN))                      # loader needs paths + modes; hashes are added at the end
STK = joblib.load(FUS/'fusion_v0_stacker.joblib'); LR, GST2, FCFG = X.load_fusion(EXP)
rg = np.random.default_rng(0); Xr = rg.normal(size=(500, 8)).astype('float32'); assert np.allclose(LR.decision_function(Xr), STK.decision_function(Xr), atol=1e-9)
say('PASS T01 the JSON logistic stacker equals the joblib stacker (500 random rows, max diff ' + f'{np.abs(LR.decision_function(Xr) - STK.decision_function(Xr)).max():.1e})')
for s in SYMBOLS:
    for m in MODELS: assert all(abs(GST2[s][m][i] - GST[s][m][i]) < 1e-12 for i in range(2))
say('PASS T02 DEV genuine statistics survive the JSON round trip')
AOF = {s: BASE/'layer1_aug_oct_v1'/f'{s.lower()}_l1_feature_matrix_aug_oct_2024.parquet' for s in SYMBOLS}; NR = 10 * 1440
REP = {}
for s in SYMBOLS:
    ao = pd.read_parquet(AOF[s]).iloc[:NR]; assert str(ao.index[0])[:10] == '2024-08-01' and str(ao.index[-1])[:10] == '2024-08-10', (ao.index[0], ao.index[-1]); REP[s] = ao.copy()
BUNX = {s: X.load_bundle_from_export(EXP, s) for s in SYMBOLS}; EXPS = {}
for s in SYMBOLS:
    F = REP[s][B.FEATS].reset_index(drop=True).copy(); F.loc[:14, 'volume_acceleration'] = 0.0
    a0 = BUN0[s].score(F); a1 = BUNX[s].score(F)
    for m in MODELS: assert np.allclose(np.nan_to_num(a0[m], nan=-1), np.nan_to_num(a1[m], nan=-1), atol=1e-6, rtol=0), (s, m)
    EXPS[s] = a1
say('PASS T03 bundles loaded from the export folder reproduce the training-folder bundles bar for bar (10 days, both symbols, all three models)')
for s in SYMBOLS:       # also equal the saved August scores (a prefix of the series)
    for m, fn in {MODELS[0]: 'ae_ctx_agg', MODELS[1]: 'xgb_v2', MODELS[2]: 'transformer_v2'}.items():
        v = np.load(OUT/'scores'/f'{fn}_{s}_{MODES[m]}.npz')['GEN_AO']; v = c.nanw(v[:, 0] if v.ndim == 2 else v)[:NR]; assert np.allclose(np.nan_to_num(EXPS[s][m], nan=-1), np.nan_to_num(v, nan=-1), atol=2e-2), (s, m)
say('PASS T04 export scores equal the saved August genuine scores (Transformer within 0.02)')

PASS T01 the JSON logistic stacker equals the joblib stacker (500 random rows, max diff 1.2e-07)
PASS T02 DEV genuine statistics survive the JSON round trip
PASS T03 bundles loaded from the export folder reproduce the training-folder bundles bar for bar (10 days, both symbols, all three models)
PASS T04 export scores equal the saved August genuine scores (Transformer within 0.02)


## 3 · Replay / golden pack (Aug 1-10, includes Aug 5)

In [ ]:
# live-style fusion: the genuine series is the stream itself, so its own scores are the reference for the rolling statistics (T1c reduces to T1)
rows = []
for s in SYMBOLS:
    sc = EXPS[s]; rs = c.ref_stats(sc, GST2[s]); Xf = c.fusion_features(sc, rs); fz = c.fusion_margin(LR, Xf)
    qa = ENGCFG['alert_q'][s]; qw = ENGCFG['watch_q']['8'][s]; qw4 = ENGCFG['watch_q']['4'][s]
    ta, tw, tw4 = c.rolling_threshold(fz, qa), c.rolling_threshold(fz, qw), c.rolling_threshold(fz, qw4)
    state = np.where(fz > ta, 'ALERT', np.where(fz > tw, 'WATCH', 'NORMAL'))
    df = pd.DataFrame({'ts': [str(t) for t in REP[s].index], 'open': REP[s]['open'].to_numpy(), 'high': REP[s]['high'].to_numpy(), 'low': REP[s]['low'].to_numpy(), 'close': REP[s]['close'].to_numpy(), 'volume': REP[s]['volume'].to_numpy()})
    for f in B.FEATS: df[f] = REP[s][f].to_numpy()
    for m, k in zip(MODELS, ['ae', 'xgb', 'tr']): df['exp_' + k] = sc[m].astype('float32')
    df['exp_fusion'] = fz.astype('float32'); df['exp_thr_alert'] = ta.astype('float32'); df['exp_thr_watch8'] = tw.astype('float32'); df['exp_thr_watch4'] = tw4.astype('float32'); df['exp_state'] = state
    CT = LR.contributions(Xf); df['exp_contrib_ae'] = (CT[:, 0] + CT[:, 5]).astype('float32'); df['exp_contrib_xgb'] = (CT[:, 1] + CT[:, 6]).astype('float32'); df['exp_contrib_tr'] = (CT[:, 2] + CT[:, 7]).astype('float32'); df['exp_contrib_shared'] = (CT[:, 3] + CT[:, 4]).astype('float32')
    def _j(a, nd=4): return [None if not np.isfinite(x) else round(float(x), nd) for x in np.asarray(a, 'float64')]
    js = {'symbol': s, 'start': df.ts.iloc[0], 'ts': list(df.ts), 'close': _j(df.close, 2), 'state': [{'NORMAL': 0, 'WATCH': 1, 'ALERT': 2}[x] for x in df.exp_state], 'fusion': _j(df.exp_fusion), 'thr_alert': _j(df.exp_thr_alert), 'thr_watch': _j(df.exp_thr_watch8), 'ae': _j(df.exp_ae), 'xgb': _j(df.exp_xgb), 'tr': _j(df.exp_tr),
          'c_ae': _j(df.exp_contrib_ae, 3), 'c_xgb': _j(df.exp_contrib_xgb, 3), 'c_tr': _j(df.exp_contrib_tr, 3), 'c_shared': _j(df.exp_contrib_shared, 3), 'alert_budget_per_day': ENGCFG['alert_budget_per_day']}
    (EXP/'replay'/f'{s}_replay.json').write_text(json.dumps(js, separators=(',', ':')))
    df.to_parquet(EXP/'replay'/f'{s}_replay_2024-08-01_to_08-10.parquet'); rows.append({'symbol': s, 'bars': len(df), 'ALERT bars': int((state == 'ALERT').sum()), 'WATCH bars': int((state == 'WATCH').sum()), 'alert episodes': B.episodes(state == 'ALERT'), 'watch-or-alert episodes': B.episodes(state != 'NORMAL')})
    d5 = np.flatnonzero(pd.to_datetime(df.ts).dt.strftime('%Y-%m-%d').to_numpy() == '2024-08-05'); st5 = pd.Series(state[d5]).value_counts().to_dict(); say(f'{s} 2024-08-05: bar states {st5}')
say(''); say(pd.DataFrame(rows).to_string(index=False)); say('(the engine must reproduce every exp_* column of these files bar for bar; this is the golden test and the replay demo data)')

BTCUSDT 2024-08-05: bar states {'NORMAL': 1427, 'ALERT': 7, 'WATCH': 6}
ETHUSDT 2024-08-05: bar states {'NORMAL': 1430, 'WATCH': 8, 'ALERT': 2}

 symbol  bars  ALERT bars  WATCH bars  alert episodes  watch-or-alert episodes
BTCUSDT 14400          45         104              20                       47
ETHUSDT 14400          47         108              20                       40
(the engine must reproduce every exp_* column of these files bar for bar; this is the golden test and the replay demo data)


## 4 · Manifest

In [ ]:
if not MOCK or True:
    files = {}
    for p in sorted(EXP.rglob('*')):
        if p.is_file() and p.name not in ('manifest.json',): files[str(p.relative_to(EXP)).replace(os.sep, '/')] = {'sha256': hashlib.sha256(p.read_bytes()).hexdigest(), 'bytes': p.stat().st_size}
    MAN.update({'created_utc': datetime.datetime.utcnow().isoformat(timespec='seconds') + 'Z', 'protocol_B2_sha256': ph, 'test_opened_utc': RES['opened_utc'], 'library_sha16': sha_file(OUT/'c03_infer.py'), 'note': 'Frozen bar models + fusion v0 for the C03 engine. Weights (*.pt, *.json under models/) go to Git LFS; do not commit data.', 'files': files, 'tgnn': None})
    (EXP/'manifest.json').write_text(json.dumps(MAN, indent=1))
big = sorted(((v['bytes'], k) for k, v in files.items()), reverse=True)[:6]; tot = sum(v['bytes'] for v in files.values())
say(f'manifest.json written: {len(files)} files, {tot / 1e6:.1f} MB; largest: ' + '; '.join(f'{k} {b / 1e6:.2f} MB' for b, k in big))
lfs = [k for k in files if k.startswith('models/') and k.endswith(('.pt', '.json')) and 'n0' not in k and 'ae_scale' not in k]; say(f'{len(lfs)} weight files for Git LFS (models/**/*.pt and models/**/xgb_*.json)')
block = '\n'.join(LINES); (OUT/'b2_step18_share_block.txt').write_text(block); print('\n===== SHARE BLOCK (attach models_b2_v1/b2_step18_share_block.txt, or copy everything below) =====\n' + block)
print('\nNEXT: copy the folder  models_b2_v1/engine_export_v1  to your PC (Drive web -> Download), then follow the folder plan I send.')

manifest.json written: 39 files, 13.8 MB; largest: replay/BTCUSDT_replay_2024-08-01_to_08-10.parquet 2.09 MB; replay/ETHUSDT_replay_2024-08-01_to_08-10.parquet 2.03 MB; replay/BTCUSDT_replay.json 1.56 MB; replay/ETHUSDT_replay.json 1.54 MB; models/ETHUSDT/xgb_s22.json 1.00 MB; models/BTCUSDT/xgb_s22.json 1.00 MB
18 weight files for Git LFS (models/**/*.pt and models/**/xgb_*.json)

===== SHARE BLOCK (attach models_b2_v1/b2_step18_share_block.txt, or copy everything below) =====
protocol B2 intact, sha256 443f99d5faf5929f; library sha256 3b5d9df23521adf9 (equals the TEST-opening library); TEST opened 2026-10-15T10:27:18Z
exported weights, N0 statistics, AE scales, fusion JSON, declared files and library to /content/drive/MyDrive/research datasets/models_b2_v1/engine_export_v1
PASS T01 the JSON logistic stacker equals the joblib stacker (500 random rows, max diff 1.2e-07)
PASS T02 DEV genuine statistics survive the JSON round trip
PASS T03 bundles loaded from the export folder reproduce 